In [0]:
%pip install requests

Python interpreter will be restarted.
Python interpreter will be restarted.


In [0]:
import requests
import os
import pandas as pd
from pandas import json_normalize
import builtins


def list_owid_datasets():
    url = 'https://api.github.com/repos/owid/owid-datasets/contents/datasets'
    response = requests.get(url)

    if response.status_code ==200:
        return response.json()
    else:
        return{'Error': f"Failed with status code {response.status_code}"}
    
data = list_owid_datasets()

# Flatten the list of GitHub items
flat_df = json_normalize(data)

#display(flat_df)


'''simple_data = [
    {
        "name": item["name"],
        "path": item["path"],
        "type": item["type"],
        "download_url": item.get("_links.html")
    }
    for item in data if isinstance(item, dict)
]

df = pd.DataFrame(simple_data)
display(df)
'''


def get_all_files(owner, repo, branch="master", extensions=None, token=None):
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
    
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"❌ Failed to fetch tree from {branch}")
        print(response.json())
        return []

    tree = response.json().get("tree", [])
    files = []

    for item in tree:
        if item["type"] == "blob":
            path = item["path"]
            if extensions and not any(path.endswith(ext) for ext in extensions):
                continue
            files.append({
                "path": path,
                "url": f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path}"
            })

    return files

DBFS_BASE_PATH = "/dbfs/FileStore/tables/Bronze/Github"

#  Replace with your actual token
token = ""  

files = get_all_files(
    owner="owid",
    repo="owid-datasets",
    branch="master",
    extensions=[".csv"],
    token=token
    
)

df = pd.DataFrame(files)
#display(df)


def get_all_csv_files(owner, repo, branch="master", token=None):
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    headers = {"Authorization": f"token {token}"} if token else {}

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(response.json())
        return []

    tree = response.json().get("tree", [])
    csv_files = []

    for item in tree:
        if item["type"] == "blob" and item["path"].endswith(".csv"):
            raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{item['path']}"
            csv_files.append({
                "name": item["path"].split("/")[-1],
                "path": item["path"],
                "url": raw_url
            })

    return csv_files


def download_and_store_in_dbfs(file_list, dbfs_base_path="/dbfs/FileStore/tables/Bronze/Github"):
    os.makedirs(dbfs_base_path, exist_ok=True)  # Create folder if it doesn't exist
    print(f"📁 Writing to: {dbfs_base_path}")
    if hasattr(builtins, 'dbutils'):
        dbutils.fs.mkdirs(dbfs_base_path)
    for file in file_list:
        try:
            response = requests.get(file["url"])
            if response.status_code == 200:
                # Handle spaces and special characters in filename (optional but safer)
                safe_name = file["name"].replace("/", "_")

# Construct path correctly for DBFS local write
                file_path = os.path.join("/dbfs/FileStore/tables/Bronze/Github", safe_name)
                os.makedirs(os.path.dirname(file_path), exist_ok=True)  # Ensure directory exists

                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(response.text)

                print(f"✅ Downloaded: {file['name']}")
            else:
                print(f"❌ Failed to download: {file['name']}")
        except Exception as e:
            print(f"⚠️ Error with {file['name']}: {e}")






# Step 1: List all CSVs
token =  ""  
csv_files = get_all_csv_files(
    owner="owid",
    repo="owid-datasets",
    branch="master",
    token=token
)


files = get_all_csv_files("owid", "owid-datasets", branch="master", token=token)
print(f"🔍 Found {len(csv_files)} CSV files.")
pd.DataFrame(csv_files).head()
# Step 2: Download to DBFS

#display(csv_files)
download_and_store_in_dbfs(csv_files,DBFS_BASE_PATH)





🔍 Found 766 CSV files.
📁 Writing to: /dbfs/FileStore/tables/Bronze/Github
✅ Downloaded: 20th century deaths in US - CDC.csv
✅ Downloaded: A Century of Work and Leisure - Ramey and Francis (2009).csv
✅ Downloaded: Absolute deaths from ambient PM2.5 air pollution- State of Global Air.csv
✅ Downloaded: Absolute population change - OWID based on HYDE & UN.csv
✅ Downloaded: Access to financial account or services (%) - World Bank (2014).csv
✅ Downloaded: Adult literacy proficiency - World Bank EdStats and STEP Skills Measurement Program.csv
✅ Downloaded: Adult obesity by region - FAO (2017).csv
✅ Downloaded: Agricultural policy support - (Agrimonitor, 2017).csv
✅ Downloaded: Agricultural total factor productivity (USDA).csv
✅ Downloaded: Air Pollutant Emissions - OECD.csv
✅ Downloaded: Air pollution by city - Fouquet and DPCC (2011).csv
✅ Downloaded: Air pollution emissions by fuel (CEDS, 2022).csv
✅ Downloaded: Air pollution emissions by fuel per capita (CEDS, 2022).csv
✅ Downloaded: Air p

In [0]:
dbutils.fs.ls("dbfs:/FileStore/tables/Bronze/Github")

Out[5]: []

In [0]:
import requests
import os
import pandas as pd
import re

# --- CONFIGURATION ---
GITHUB_TOKEN = ""  # Replace with your token or use "" for public
OWNER = "owid"
REPO = "owid-datasets"
BRANCH = "master"
DBFS_BASE_PATH = "/dbfs/FileStore/tables/Bronze/Github/"

# --- STEP 1: Get All CSV Files from Repo Tree ---
def get_all_csv_files(owner, repo, branch="master", token=None):
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    headers = {"Authorization": f"token {token}"} if token else {}

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"❌ GitHub API failed with status {response.status_code}")
        print(response.json())
        return []

    tree = response.json().get("tree", [])
    csv_files = []

    for item in tree:
        if item["type"] == "blob" and item["path"].endswith(".csv"):
            csv_files.append({
                "name": item["path"].split("/")[-1],
                "path": item["path"],
                "url": f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{item['path']}"
            })

    return csv_files

# --- STEP 2: Sanitize Filenames ---
def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)

# --- STEP 3: Download and Write to DBFS ---
def download_and_store_in_dbfs(file_list, dbfs_base_path):
    os.makedirs(dbfs_base_path, exist_ok=True)
    print(f"📁 Writing to: {dbfs_base_path}")

    for file in file_list:
        try:
            response = requests.get(file["url"])
            if response.status_code == 200:
                safe_name = sanitize_filename(file["name"])
                file_path = os.path.join(dbfs_base_path, safe_name)

                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(response.text)

                print(f"✅ Saved: {file_path}")
            else:
                print(f"❌ Failed to download {file['name']}: HTTP {response.status_code}")
        except Exception as e:
            print(f"⚠️ Error saving {file['name']}: {e}")

# --- RUN SCRIPT ---
csv_files = get_all_csv_files(OWNER, REPO, BRANCH, token=GITHUB_TOKEN)
print(f"🔍 Found {len(csv_files)} CSV files.")

if csv_files:
    download_and_store_in_dbfs(csv_files, DBFS_BASE_PATH)
else:
    print("⚠️ No CSV files found. Check GitHub token or API limits.")


🔍 Found 766 CSV files.
📁 Writing to: /dbfs/FileStore/tables/Bronze/Github/
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/20th century deaths in US - CDC.csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/A Century of Work and Leisure - Ramey and Francis (2009).csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Absolute deaths from ambient PM2.5 air pollution- State of Global Air.csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Absolute population change - OWID based on HYDE & UN.csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Access to financial account or services (%) - World Bank (2014).csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Adult literacy proficiency - World Bank EdStats and STEP Skills Measurement Program.csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Adult obesity by region - FAO (2017).csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Agricultural policy support - (Agrimonitor, 2017).csv
✅ Saved: /dbfs/FileStore/tables/Bronze/Github/Agricultural total factor produc